In [32]:
!pip install yfinance --upgrade --no-cache-dir

In [33]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [34]:
def compute_macd(
    df: pd.DataFrame,
    fast: int = 12,
    slow: int = 26,
    signal: int = 9,
    price_col: str | None = None,
):
    """
    Compute MACD, Signal line, and Histogram.

    Parameters
    ----------
    df : pd.DataFrame
        OHLCV dataframe
    fast : int
        Fast EMA period (default 12)
    slow : int
        Slow EMA period (default 26)
    signal : int
        Signal EMA period (default 9)
    price_col : str | None
        Column to use as price. If None, auto-detects 'close' or 'Close'.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        - macd
        - macd_signal
        - macd_hist
    """

    # ---- Detect price column safely ----
    if price_col is None:
        if "close" in df.columns:
            price = df["close"]
        elif "Close" in df.columns:
            price = df["Close"]
        else:
            raise KeyError("No 'close' or 'Close' column found in dataframe")
    else:
        if price_col not in df.columns:
            raise KeyError(f"'{price_col}' not found in dataframe")
        price = df[price_col]

    # ---- EMAs ----
    ema_fast = price.ewm(span=fast, adjust=False).mean()
    ema_slow = price.ewm(span=slow, adjust=False).mean()

    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal, adjust=False).mean()
    macd_hist = macd - macd_signal

    df["macd"] = macd
    df["macd_signal"] = macd_signal

In [35]:
def add_rsi(
    df: pd.DataFrame,
    period: int = 14,
    price_col: str | None = None,
    col_name: str = "rsi",
) -> pd.DataFrame:
    """
    Calculate Wilder's RSI and add it as a column to the dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        OHLCV dataframe
    period : int
        RSI period (default 14)
    price_col : str | None
        Column to use as price. If None, auto-detects 'close' or 'Close'.
    col_name : str
        Name of RSI column to add

    Returns
    -------
    pd.DataFrame
        Original dataframe with RSI column added
    """

    # ---- Detect price column ----
    if price_col is None:
        if "close" in df.columns:
            price = df["close"]
        elif "Close" in df.columns:
            price = df["Close"]
        else:
            raise KeyError("No 'close' or 'Close' column found in dataframe")
    else:
        if price_col not in df.columns:
            raise KeyError(f"'{price_col}' not found in dataframe")
        price = df[price_col]

    # ---- Price differences ----
    delta = price.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # ---- Wilder smoothing (EMA with alpha = 1/period) ----
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    df[col_name] = rsi
    return df

In [36]:
def make_features(df: pd.DataFrame):
  df = df.copy()
  df = df[df["Volume"]>0]
  df["rel_volume"] = df["Volume"] / df["Volume"].rolling(20).mean()
  df["hl_range"] = (df["High"] - df["Low"]) / df["Close"].shift(1)
  df["oc_gap"]   = (df["Open"] - df["Close"].shift(1)) / df["Close"].shift(1)
  df["sma_5"]  = df["Close"].rolling(5).mean()
  df["sma_10"] = df["Close"].rolling(10).mean()
  df["sma_ratio"] = df["sma_5"] / df["sma_10"] - 1.0
  df["Direction"] = (df["Close"].shift(-1) > df["Close"]).astype(int)
  return df

In [37]:
model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500, class_weight="balanced"))
    ])

In [38]:
ircon= yf.Ticker("IRCON.NS")
ircon_history = ircon.history(start="2025-10-01", end="2025-12-31", interval="1d")
train = ircon_history.drop(columns = ["Dividends","Stock Splits"])
train = make_features(train)
compute_macd(train)
add_rsi(train)
train.drop(train.index[0:21], inplace=True)

FEATURE_COLS = ["macd","macd_signal","rsi","rel_volume","hl_range","sma_ratio","oc_gap"]
X_train = train[FEATURE_COLS]
y_train = train["Direction"]

model.fit(X_train,y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=500))])

In [39]:
ircon_test = ircon.history(start="2025-12-04", end="2026-01-08", interval="1d")
test = ircon_test.drop(columns = ["Dividends","Stock Splits"])
test = make_features(test)
compute_macd(test)
add_rsi(test)
test.drop(test.index[0:19], inplace=True)

X_test = test[FEATURE_COLS]
y_test = test["Direction"]

In [40]:
predictions = model.predict(X_train)
train["Prediction"] = predictions
Prob = model.predict_proba(X_train)
Up_Prob = np.zeros(len((X_train)))

for i, prob in enumerate(Prob):
  Up_Prob[i] = prob[1]

train["Up_Prob"] = Up_Prob

In [41]:
predictions = model.predict(X_test)
test["Prediction"] = predictions
Prob = model.predict_proba(X_test)
Up_Prob = np.zeros(len((X_test)))

for i, prob in enumerate(Prob):
  Up_Prob[i] = prob[1]

test["Up_Prob"] = Up_Prob
test

,Open,High,Low,Close,Volume,rel_volume,hl_range,oc_gap,sma_5,sma_10,sma_ratio,Direction,macd,macd_signal,rsi,Prediction,Up_Prob
Date,,,,,,,,,,,,,,,,,
2026-01-01 00:00:00+05:30,177.600006,179.690002,175.509995,177.979996,3712327,0.401516,0.023529,-0.000281,175.720001,167.837000,0.046968,1,5.953706,3.606598,58.332159,0,0.139065
2026-01-02 00:00:00+05:30,178.770004,182.369995,177.910004,179.220001,6334893,0.665581,0.025059,0.004439,175.822000,170.758000,0.029656,0,6.387686,4.162816,59.413479,0,0.200724
2026-01-05 00:00:00+05:30,179.220001,180.800003,176.220001,177.020004,2913926,0.303329,0.025555,0.000000,176.672000,173.165001,0.020252,1,6.479408,4.626134,56.606726,0,0.231714
2026-01-06 00:00:00+05:30,177.399994,181.529999,175.800003,177.259995,4488652,0.461327,0.032369,0.002147,177.825998,175.210001,0.014931,1,6.496574,5.000222,56.846220,0,0.242811
2026-01-07 00:00:00+05:30,177.259995,179.449997,175.860001,177.419998,2786316,0.286777,0.020253,0.000000,177.779999,176.022000,0.009987,0,6.448752,5.289928,57.016551,0,0.269447


In [42]:
STOP_LOSS = -0.02        # -2%
TAKE_PROFIT = 0.04       # +4%
TRANSACTION_COST = 0.0015

In [43]:
def backtest(df, buy_th, sell_th):
    position = 0
    entry_price = 0
    returns = []

    for i in range(1, len(df)):
        price = df.loc[i, "Close"]
        prev_price = df.loc[i - 1, "Close"]
        up_prob = df.loc[i, "Up_Prob"]

        # ---- CORE SIGNAL (UNCHANGED) ----
        if up_prob >= buy_th:
            signal = 1
        elif up_prob <= sell_th:
            signal = -1
        else:
            signal = 0

        daily_ret = 0

        # ---- POSITION MANAGEMENT ----
        if position == 1:
            pnl = (price - entry_price) / entry_price

            if pnl <= STOP_LOSS or pnl >= TAKE_PROFIT or signal == -1:
                daily_ret = pnl - TRANSACTION_COST
                position = 0
                entry_price = 0
            else:
                daily_ret = (price - prev_price) / prev_price

        elif position == 0 and signal == 1:
            position = 1
            entry_price = price
            daily_ret -= TRANSACTION_COST

        returns.append(daily_ret)

    returns = np.array(returns)
    equity = np.cumprod(1 + returns)

    sharpe = np.sqrt(252) * returns.mean() / (returns.std() + 1e-9)
    max_dd = np.min(equity / np.maximum.accumulate(equity) - 1)
    total_return = equity[-1] - 1

    return sharpe, total_return, max_dd

In [44]:
results = []

for buy_th in np.arange(0.55, 0.76, 0.02):
    for sell_th in np.arange(0.25, 0.46, 0.02):
        if sell_th >= buy_th:
            continue

        sharpe, ret, dd = backtest(train, buy_th, sell_th)

        results.append({
            "BUY_TH": buy_th,
            "SELL_TH": sell_th,
            "Sharpe": sharpe,
            "Return": ret,
            "MaxDD": dd
        })

results_df = pd.DataFrame(results)

best = results_df.sort_values("Sharpe", ascending=False).iloc[0]

BUY_TH_OPT = best["BUY_TH"]
SELL_TH_OPT = best["SELL_TH"]

print("\n✅ OPTIMAL THRESHOLDS (TRAIN)")
print(f"BUY  ≥ {BUY_TH_OPT:.2f}")
print(f"SELL ≤ {SELL_TH_OPT:.2f}")
print(f"Sharpe: {best['Sharpe']:.2f}")
print(f"Return: {best['Return']:.2%}")
print(f"MaxDD : {best['MaxDD']:.2%}")

KeyError: 1

In [ ]:
test_sharpe, test_ret, test_dd = backtest(
    test,
    BUY_TH_OPT,
    SELL_TH_OPT
)

print("\n📊 FINAL OUT-OF-SAMPLE PERFORMANCE (TEST)")
print(f"Total Return : {test_ret:.2%}")
print(f"Max Drawdown : {test_dd:.2%}")
print(f"Sharpe Ratio : {test_sharpe:.2f}")